# SocialChoice 01 : Théorème d'impossibilité d'Arrow (twin C# .NET)

> **Twin C# .NET de `GameTheory/SocialChoice/01-Arrow-Impossibility-Theorem.ipynb`** (Python).
> Marathon **#4956** (jumeaux .NET ⇄ Python), Prong B (#3801) : implémentations **from-scratch** (BCL .NET 9, **0 NuGet**).

**Théorème d'Arrow (1951).** Aucune règle d'agrégation ne satisfait simultanément les trois axiomes :
1. **Universalité** (domaine non restreint) — traitée implicitement (on énumère tous les profils) ;
2. **Pareto faible** (unanimité) — si tous préfèrent $x \succ y$, le social aussi ;
3. **Indépendance aux alternatives non pertinentes (IIA)** — le classement social entre $x$ et $y$ ne dépend que des préférences individuelles entre $x$ et $y$ ;
4. **Non-dictature** — aucun votant n'impose toujours son classement.

(Arrow compte Universalité + 3 axiomes ; on vérifie ici Pareto, IIA, Non-dictature.)

### ★ Leçon de parité : déterministe > stochastique

Le notebook Python teste les axiomes par **simulation aléatoire** (`random.shuffle`, `seed(42)`). Cela pose deux problèmes :
- (a) le RNG Python et C# **divergent** → pas de parité byte-pour-byte possible ;
- (b) un test stochastique **rate** les violations rares et reste opaque sur *quels* profils violent l'axiome.

Le twin C# remplace la simulation par **l'énumération exhaustive déterministe** : pour 3 alternatives et 3 votants, il n'y a que $(3!)^3 = 216$ profils — on peut tous les parcourir et **compter exactement** les violations, puis **afficher un contre-exemple concret**. C'est à la fois plus rigoureux (preuve exhaustive) et plus pédagogique (on voit le profil coupable). Voir leçon c.96 (démo IIA déterministe du twin SC-03).

**Plan :**
1. Règles de vote (Borda, Pluralité, Dictatoriale) — from-scratch ;
2. Vérification des 3 axiomes (Pareto, IIA, Non-dictature) — déterministe ;
3. Structure de la preuve d'Arrow (lemme extremal, pivot, dictateur partiel) ;
4. Réfutation par force brute (énumération complète) + synthèse.


## 0. Configuration

In [1]:
// SocialChoice 01 : Théorème d'impossibilité d'Arrow -- twin C# de 01-Arrow
// Prong B (#3801) : implementations from-scratch (BCL .NET 9, 0 NuGet).
// Lecon c.94 : les 3 setters de culture sont requis (CurrentCulture seul ne
// persiste pas cross-cell, threads du pool distincts).
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;

Console.WriteLine("Configuration OK : SocialChoice 01 - Arrow (twin C# .NET)");


Configuration OK : SocialChoice 01 - Arrow (twin C# .NET)


## 1. Règles de vote (from-scratch)

Un **profil** est une liste de classements (un par votant, du meilleur au pire).
On implémente trois règles d'agrégation :

- **Borda** : points $(n-1-\text{rang})$ par votant, classement par score décroissant ;
- **Pluralité** : compte les premiers choix, classement par nombre de premiers choix ;
- **Dictatoriale** : le classement social = préférence du dictateur (indice fixé).


In [2]:
// === Section 1 : Regles de vote (Borda / Pluralite / Dictatoriale) ===
// Un profil = liste des classements (meilleur -> pire) de chaque votant.
// Une regle prend un profil et retourne un classement social (meilleur -> pire).

public static List<string> Borda(List<List<string>> profile)
{
    var scores = new Dictionary<string, int>();
    int n = profile[0].Count;
    foreach (var pref in profile)
        for (int rank = 0; rank < n; rank++)
        {
            if (!scores.ContainsKey(pref[rank])) scores[pref[rank]] = 0;
            scores[pref[rank]] += (n - 1 - rank);
        }
    // Tri par score decroissant ; ex-aequo par ordre alpha pour determinisme
    return scores.Keys.OrderByDescending(k => scores[k]).ThenBy(k => k).ToList();
}

public static List<string> Pluralite(List<List<string>> profile)
{
    // Tiebreak par ordre d'apparition dans le profil (stable, comme Python
    // dict.fromkeys) -- sinon deux alternatives a 0 premier-choix seraient
    // ordonnees alphabetiquement et violeraient Pareto artificiellement.
    var first = new Dictionary<string, int>();
    var appearance = new List<string>();
    foreach (var pref in profile)
        foreach (var a in pref)
        {
            if (!appearance.Contains(a)) appearance.Add(a);
            if (!first.ContainsKey(a)) first[a] = 0;
        }
    foreach (var pref in profile) first[pref[0]]++;
    return appearance.OrderBy(k => -first[k]).ThenBy(k => appearance.IndexOf(k)).ToList();
}

public static List<string> Dictatorial(List<List<string>> profile, int dictatorIdx)
    => new List<string>(profile[dictatorIdx]);

// Demo : profil a 3 votants, 3 alternatives
var demo = new List<List<string>> {
    new() { "A", "B", "C" },
    new() { "B", "A", "C" },
    new() { "A", "C", "B" },
};
Console.WriteLine("Profil demo :");
for (int i = 0; i < demo.Count; i++)
    Console.WriteLine("  Votant " + i + " : " + string.Join(" > ", demo[i]));
Console.WriteLine();
Console.WriteLine("Borda       : " + string.Join(" > ", Borda(demo)));
Console.WriteLine("Pluralite   : " + string.Join(" > ", Pluralite(demo)));
Console.WriteLine("Dictatorial : " + string.Join(" > ", Dictatorial(demo, 0)));


Profil demo :


  Votant 0 : A > B > C


  Votant 1 : B > A > C


  Votant 2 : A > C > B


Borda       : A > B > C


Pluralite   : A > B > C


Dictatorial : A > B > C


### 1.1 Axiome de Pareto faible (déterministe)

**Pareto faible** : si **tous** les votants préfèrent $x \succ y$, alors le classement social doit aussi avoir $x$ avant $y$.

Plutôt qu'un tirage aléatoire, on **énumère tous les profils** possibles (pour 3 alternatives, 3 votants : $(3!)^3 = 216$ profils) et on compte les violations de Pareto. Une règle satisfaisant Pareto donne **0 violation**.


In [3]:
// === Section 1.1 : Pareto faible -- enumeration exhaustive deterministe ===
// Genere toutes les permutations de 'alts' (classements possibles d'un votant).
public static List<List<string>> AllPermutations(List<string> alts)
{
    var result = new List<List<string>>();
    int n = alts.Count;
    int[] idx = Enumerable.Range(0, n).ToArray();
    // Permutations par algorithme de Heap (deterministe)
    void Swap(int i, int j) { (idx[i], idx[j]) = (idx[j], idx[i]); }
    void Heap(int k)
    {
        if (k == 1) { result.Add(idx.Select(i => alts[i]).ToList()); return; }
        for (int i = 0; i < k; i++) { Heap(k - 1); Swap(i % 2 == 0 ? 0 : i, k - 1); }
    }
    Heap(n);
    return result;
}

// Genere tous les profils a nVotants (produit cartesien des permutations).
public static List<List<List<string>>> AllProfiles(List<string> alts, int nVotants)
{
    var perms = AllPermutations(alts);
    var profiles = new List<List<List<string>>>();
    // Recursion : produit cartesien
    void Build(int remaining, List<List<string>> current)
    {
        if (remaining == 0) { profiles.Add(current.Select(x => x).ToList()); return; }
        foreach (var p in perms)
        {
            current.Add(p);
            Build(remaining - 1, current);
            current.RemoveAt(current.Count - 1);
        }
    }
    Build(nVotants, new List<List<string>>());
    return profiles;
}

// Predicat : tous les votants preferent x a y ?
public static bool AllPrefer(List<List<string>> profile, string x, string y)
    => profile.All(p => p.IndexOf(x) < p.IndexOf(y));

// Compte les violations de Pareto faible sur TOUS les profils.
public static int ParetoViolations(
    Func<List<List<string>>, List<string>> rule,
    List<string> alts, int nVotants)
{
    int violations = 0;
    var profiles = AllProfiles(alts, nVotants);
    foreach (var prof in profiles)
    {
        var ranking = rule(prof);
        // Toutes les paires non ordonnees {x, y} avec x != y
        for (int i = 0; i < alts.Count; i++)
            for (int j = 0; j < alts.Count; j++)
            {
                if (i == j) continue;
                string x = alts[i], y = alts[j];
                if (AllPrefer(prof, x, y) && ranking.IndexOf(x) > ranking.IndexOf(y))
                    violations++;
            }
    }
    return violations;
}

var alts3 = new List<string> { "A", "B", "C" };
int nVot = 3;
int np = AllProfiles(alts3, nVot).Count;
Console.WriteLine("Enumeration exhaustive : " + np + " profils (" + alts3.Count + " alts, " + nVot + " votants)");
Console.WriteLine();
Console.WriteLine("Violations de Pareto faible (devrait etre 0 pour Borda/Pluralite/Dictatorial) :");
Console.WriteLine("  Borda       : " + ParetoViolations(p => Borda(p), alts3, nVot));
Console.WriteLine("  Pluralite   : " + ParetoViolations(p => Pluralite(p), alts3, nVot));
Console.WriteLine("  Dictatorial : " + ParetoViolations(p => Dictatorial(p, 0), alts3, nVot));


Enumeration exhaustive : 216 profils (3 alts, 3 votants)


Violations de Pareto faible (devrait etre 0 pour Borda/Pluralite/Dictatorial) :


  Borda       : 0


  Pluralite   : 0


  Dictatorial : 0


### 1.2 Axiome d'Indépendance (IIA) — contre-exemple déterministe

**IIA** : le classement social entre $x$ et $y$ ne dépend que des préférences individuelles entre $x$ et $y$ (les autres alternatives ne doivent pas influencer).

**Borda viole IIA** — on le montre par un contre-exemple **construit et déterministe** (leçon c.96) : deux profils où les préférences A-vs-B sont **identiques** pour chaque votant, mais le classement social Borda de A-vs-B **bascule**.

| Profil 1 (3×A>B>C, 2×B>C>A) | Profil 2 (3×A>C>B, 2×B>A>C) |
|---|---|
| Votants 1-3 préfèrent A>B | Votants 1-3 préfèrent A>B |
| Votants 4-5 préfèrent B>A | Votants 4-5 préfèrent B>A |

→ Préférences A-vs-B **individuelles identiques**. Pourtant Borda donne Profil 1 : B>A, Profil 2 : A>B. **Violation de IIA**.


In [4]:
// === Section 1.2 : IIA -- contre-exemple construit deterministe (lecon c.96) ===
// Deux profils ou les preferences A-vs-B individuelles sont IDENTIQUES,
// mais le classement social Borda de A-vs-B bascule = violation de IIA.

var iiaProfil1 = new List<List<string>> {
    new() { "A", "B", "C" }, new() { "A", "B", "C" }, new() { "A", "B", "C" },
    new() { "B", "C", "A" }, new() { "B", "C", "A" },
};
var iiaProfil2 = new List<List<string>> {
    new() { "A", "C", "B" }, new() { "A", "C", "B" }, new() { "A", "C", "B" },
    new() { "B", "A", "C" }, new() { "B", "A", "C" },
};

static int BordaScore(List<List<string>> profile, string alt)
{
    int n = profile[0].Count, s = 0;
    foreach (var p in profile) s += (n - 1 - p.IndexOf(alt));
    return s;
}

int a1 = BordaScore(iiaProfil1, "A"), b1 = BordaScore(iiaProfil1, "B"), c1 = BordaScore(iiaProfil1, "C");
int a2 = BordaScore(iiaProfil2, "A"), b2 = BordaScore(iiaProfil2, "B"), c2 = BordaScore(iiaProfil2, "C");
Console.WriteLine("Profil 1 (3xA>B>C, 2xB>C>A) : scores Borda A=" + a1 + " B=" + b1 + " C=" + c1
                  + "  -> " + (b1 > a1 ? "B > A" : "A > B"));
Console.WriteLine("Profil 2 (3xA>C>B, 2xB>A>C) : scores Borda A=" + a2 + " B=" + b2 + " C=" + c2
                  + "  -> " + (a2 > b2 ? "A > B" : "B > A"));
Console.WriteLine();

// Verification : preferences A-vs-B individuelles identiques entre les 2 profils ?
bool SameAB(List<List<string>> p1, List<List<string>> p2)
    => p1.Zip(p2).All(t => (t.First.IndexOf("A") < t.First.IndexOf("B"))
                        == (t.Second.IndexOf("A") < t.Second.IndexOf("B")));
Console.WriteLine("Preferences A-vs-B identiques entre Profil 1 et Profil 2 ? " + SameAB(iiaProfil1, iiaProfil2));
Console.WriteLine("Verdict social Profil 1 : " + (b1 > a1 ? "B > A" : "A > B"));
Console.WriteLine("Verdict social Profil 2 : " + (a2 > b2 ? "A > B" : "B > A"));
Console.WriteLine(">>> BORDA VIOLE IIA : les preferences A-vs-B sont les memes, pourtant le verdict bascule.");


Profil 1 (3xA>B>C, 2xB>C>A) : scores Borda A=6 B=7 C=2  -> B > A


Profil 2 (3xA>C>B, 2xB>A>C) : scores Borda A=8 B=4 C=3  -> A > B


Preferences A-vs-B identiques entre Profil 1 et Profil 2 ? True


Verdict social Profil 1 : B > A


Verdict social Profil 2 : A > B


>>> BORDA VIOLE IIA : les preferences A-vs-B sont les memes, pourtant le verdict bascule.


### 1.2 (suite) — Comptage exhaustif des violations IIA

On confirme par **énumération exhaustive** : pour chaque paire de profils $(P_1, P_2)$ parmi les 216, et chaque paire d'alternatives $(x,y)$, si les préférences individuelles $x$-vs-$y$ coïncident mais que le verdict social diffère → 1 violation. On affiche le nombre total et le **premier contre-exemple trouvé**.


In [5]:
// === Section 1.2 (suite) : comptage exhaustif des violations IIA ===
public static (int count, string first) IiaViolations(
    Func<List<List<string>>, List<string>> rule,
    List<string> alts, int nVotants)
{
    var profiles = AllProfiles(alts, nVotants);
    int count = 0;
    string first = null;
    foreach (var p1 in profiles)
    {
        var r1 = rule(p1);
        foreach (var p2 in profiles)
        {
            var r2 = rule(p2);
            for (int i = 0; i < alts.Count; i++)
                for (int j = i + 1; j < alts.Count; j++)
                {
                    string x = alts[i], y = alts[j];
                    // Preferences individuelles x-vs-y identiques entre P1 et P2 ?
                    bool sameXY = p1.Zip(p2).All(t =>
                        (t.First.IndexOf(x) < t.First.IndexOf(y))
                     == (t.Second.IndexOf(x) < t.Second.IndexOf(y)));
                    if (!sameXY) continue;
                    bool soc1 = r1.IndexOf(x) < r1.IndexOf(y);
                    bool soc2 = r2.IndexOf(x) < r2.IndexOf(y);
                    if (soc1 != soc2)
                    {
                        count++;
                        if (first == null)
                            first = "x=" + x + " y=" + y + " : P1 social "
                                 + (soc1 ? (x + ">" + y) : (y + ">" + x))
                                 + " vs P2 social " + (soc2 ? (x + ">" + y) : (y + ">" + x));
                    }
                }
        }
    }
    return (count, first ?? "(aucune)");
}

var (viiaB, firstB) = IiaViolations(p => Borda(p), alts3, nVot);
var (viiaP, firstP) = IiaViolations(p => Pluralite(p), alts3, nVot);
Console.WriteLine("Violations IIA (enumeration exhaustive, 216^2 paires de profils) :");
Console.WriteLine("  Borda       : " + viiaB + " violations");
Console.WriteLine("    Premier exemple : " + firstB);
Console.WriteLine("  Pluralite   : " + viiaP + " violations");
Console.WriteLine("    Premier exemple : " + firstP);


Violations IIA (enumeration exhaustive, 216^2 paires de profils) :


  Borda       : 492 violations


    Premier exemple : x=A y=B : P1 social B>A vs P2 social A>B


  Pluralite   : 2568 violations


    Premier exemple : x=A y=B : P1 social A>B vs P2 social B>A


### 1.3 Axiome de Non-dictature (déterministe)

**Non-dictature** : aucun votant $d$ n'est un dictateur, i.e. dont le classement social coïnciderait **toujours** avec sa préférence stricte (pour toute paire $x,y$, si $d$ préfère $x \succ y$ alors le social a $x$ avant $y$).

On teste **déterministiquement** chaque votant sur les 216 profils : si un votant dicte toutes les paires dans tous les profils, c'est un dictateur.


In [6]:
// === Section 1.3 : Non-dictature -- enumeration exhaustive deterministe ===
// Retourne l'index du dictateur, ou -1 si aucun.
public static int FindDictator(
    Func<List<List<string>>, List<string>> rule,
    List<string> alts, int nVotants)
{
    var profiles = AllProfiles(alts, nVotants);
    for (int d = 0; d < nVotants; d++)
    {
        bool isDictator = true;
        foreach (var prof in profiles)
        {
            var ranking = rule(prof);
            for (int i = 0; i < alts.Count; i++)
                for (int j = i + 1; j < alts.Count; j++)
                {
                    string x = alts[i], y = alts[j];
                    if (prof[d].IndexOf(x) < prof[d].IndexOf(y)
                        && ranking.IndexOf(x) > ranking.IndexOf(y))
                    { isDictator = false; break; }
                    if (prof[d].IndexOf(y) < prof[d].IndexOf(x)
                        && ranking.IndexOf(y) > ranking.IndexOf(x))
                    { isDictator = false; break; }
                }
            if (!isDictator) break;
        }
        if (isDictator) return d;
    }
    return -1;
}

int dBorda = FindDictator(p => Borda(p), alts3, nVot);
int dPlur = FindDictator(p => Pluralite(p), alts3, nVot);
int dDict0 = FindDictator(p => Dictatorial(p, 0), alts3, nVot);
Console.WriteLine("Dictateur detecte (-1 = aucun) :");
Console.WriteLine("  Borda       : " + (dBorda < 0 ? "AUCUN (satisfait Non-dictature)" : "Votant " + dBorda));
Console.WriteLine("  Pluralite   : " + (dPlur < 0 ? "AUCUN (satisfait Non-dictature)" : "Votant " + dPlur));
Console.WriteLine("  Dictatorial : " + (dDict0 < 0 ? "AUCUN" : "Votant " + dDict0 + " (VIOLE Non-dictature par construction)"));


Dictateur detecte (-1 = aucun) :


  Borda       : AUCUN (satisfait Non-dictature)


  Pluralite   : AUCUN (satisfait Non-dictature)


  Dictatorial : Votant 0 (VIOLE Non-dictature par construction)


In [7]:
// === Section 2.1 : Lemme extremal (demo construite) ===
// On construit un profil ou 'B' est TOUJOURS en position extreme (premier ou dernier)
// et on verifie que Borda le place aussi en position extreme.

var extremal = new List<List<string>> {
    new() { "A", "C", "B" },  // B dernier
    new() { "B", "A", "C" },  // B premier
    new() { "C", "A", "B" },  // B dernier
    new() { "B", "C", "A" },  // B premier
    new() { "A", "C", "B" },  // B dernier
};
var rankExt = Borda(extremal);
int posB = rankExt.IndexOf("B");
bool isExtreme = (posB == 0 || posB == rankExt.Count - 1);
Console.WriteLine("Lemme extremal : profil ou B est toujours en position extreme");
Console.WriteLine("  Classement Borda : " + string.Join(" > ", rankExt));
Console.WriteLine("  Position de B    : " + posB + " (sur " + (rankExt.Count - 1) + ")");
Console.WriteLine("  B en position extreme dans le social ? " + isExtreme);
Console.WriteLine("  (Sous Pareto + IIA, le lemme extremal garantit cela pour toute regle admissible.)");


Lemme extremal : profil ou B est toujours en position extreme


  Classement Borda : A > C > B


  Position de B    : 2 (sur 2)


  B en position extreme dans le social ? True


  (Sous Pareto + IIA, le lemme extremal garantit cela pour toute regle admissible.)


In [8]:
// === Section 2.2 : Existence du pivot (demo construite) ===
// On part d'un profil ou B est dernier pour tous, puis on bascule B en premier
// pour les votants un par un. On cherche l'electeur pivot dont le bascule fait
// passer B de dernier a premier dans le classement Borda.

var others = new List<string> { "A", "C" };
int nv = 5;
// Etat initial : tous placent B dernier (ordre A,C determine)
var pivotProf = new List<List<string>>();
for (int i = 0; i < nv; i++) pivotProf.Add(new List<string> { (i % 2 == 0 ? "A" : "C"), (i % 2 == 0 ? "C" : "A"), "B" });

int pivotIdx = -1;
for (int step = 0; step < nv; step++)
{
    // L'electeur 'step' bascule B en premier
    pivotProf[step] = new List<string> { "B", (step % 2 == 0 ? "A" : "C"), (step % 2 == 0 ? "C" : "A") };
    var r = Borda(pivotProf);
    if (r.IndexOf("B") == 0) { pivotIdx = step; break; }
}
Console.WriteLine("Recherche du pivot (B bascule du bas vers le sommet, votant par votant) :");
Console.WriteLine("  Electeur pivot detecte : " + (pivotIdx >= 0 ? "Votant " + pivotIdx : "AUCUN"));
Console.WriteLine("  (Sous Pareto + IIA, l'existence d'un tel pivot est garantie des qu'il y a >= 3 alternatives.)");


Recherche du pivot (B bascule du bas vers le sommet, votant par votant) :


  Electeur pivot detecte : Votant 2


  (Sous Pareto + IIA, l'existence d'un tel pivot est garantie des qu'il y a >= 3 alternatives.)


## 2. Structure de la preuve d'Arrow

La preuve formelle d'Arrow (pour $\geq 3$ alternatives) procède en trois temps. On l'illustre ici **déterministiquement** sur la règle de Borda :

1. **Lemme extremal** : si une alternative $b$ est **toujours** placée en position extrême (première ou dernière) par chaque votant, alors elle est aussi en position extrême dans le classement social (sous Pareto + IIA).
2. **Existence du pivot** : en faisant basculer $b$ du bas vers le sommet, électeur par électeur, il existe un électeur **pivot** dont le basculement fait passer $b$ de dernier à premier.
3. **Le pivot est dictateur partiel** : ce pivot dicte alors le classement pour toute paire ne contenant pas $b$.


In [9]:
// === Section 2.3 : Le pivot est dictateur partiel (demo) ===
// Pour toute paire (a, c) ne contenant pas la cible b, le pivot dicte le classement.
// On l'illustre : le pivot detecte prefere-t-il A a C ?
// Le classement social Borda suit-il cette preference ?

if (pivotIdx >= 0)
{
    // On reconstruit un profil ou le pivot est libre sur A vs C
    var testProf = new List<List<string>> {
        new() { "A", "C", "B" }, new() { "C", "A", "B" }, new() { "A", "C", "B" },
        new() { "C", "A", "B" }, new() { "A", "C", "B" },
    };
    var r = Borda(testProf);
    bool pivotPrefA = testProf[pivotIdx].IndexOf("A") < testProf[pivotIdx].IndexOf("C");
    bool socialPrefA = r.IndexOf("A") < r.IndexOf("C");
    Console.WriteLine("Pivot = Votant " + pivotIdx);
    Console.WriteLine("  Le pivot prefere A a C ? " + pivotPrefA);
    Console.WriteLine("  Le social (Borda) prefere A a C ? " + socialPrefA);
    Console.WriteLine("  Concordance pivot-social sur (A,C) ? " + (pivotPrefA == socialPrefA));
    Console.WriteLine("  (Sous Pareto + IIA, cette concordance vaut pour TOUTE paire sans b : le pivot est dictateur partiel.)");
}
else
{
    Console.WriteLine("Pas de pivot detecte dans la demo precedente (rare ; Borda y viole deja IIA).");
}


Pivot = Votant 2


  Le pivot prefere A a C ? True


  Le social (Borda) prefere A a C ? True


  Concordance pivot-social sur (A,C) ? True


  (Sous Pareto + IIA, cette concordance vaut pour TOUTE paire sans b : le pivot est dictateur partiel.)


## 3. Réfutation par force brute (énumération complète)

On rassemble les trois axiomes sur **l'ensemble des 216 profils** et on dresse le tableau synthétique. Le théorème d'Arrow prédit : **aucune** règle ne satisfait simultanément Pareto + IIA + Non-dictature.

| Règle | Pareto | IIA | Non-dictature |
|-------|--------|-----|---------------|
| Borda | ✓ (0 violation) | ✗ (violations) | ✓ (aucun dictateur) |
| Pluralité | ✓ | ✗ | ✓ |
| Dictatoriale | ✓ | ✓ | ✗ (le dictateur) |


In [10]:
// === Section 3 : Force brute -- synthese exhaustive des 3 axiomes ===
Console.WriteLine("Refutation par force brute (216 profils, 3 alternatives, 3 votants)");
Console.WriteLine(new string('-', 62));
Console.WriteLine(string.Format("{0,-13} {1,-10} {2,-22} {3,-15}", "Regle", "Pareto", "IIA (violations)", "Non-dictature"));
Console.WriteLine(new string('-', 62));

string Row(string name, Func<List<List<string>>, List<string>> rule)
{
    int par = ParetoViolations(rule, alts3, nVot);
    var (viia, _) = IiaViolations(rule, alts3, nVot);
    int dic = FindDictator(rule, alts3, nVot);
    string paretoTxt = par == 0 ? "OK" : ("ECHEC x" + par);
    string iiaTxt = viia == 0 ? "OK" : ("ECHEC x" + viia);
    string dicTxt = dic < 0 ? "OK" : ("Dict(V" + dic + ")");
    return string.Format("{0,-13} {1,-10} {2,-22} {3,-15}", name, paretoTxt, iiaTxt, dicTxt);
}

Console.WriteLine(Row("Borda", p => Borda(p)));
Console.WriteLine(Row("Pluralite", p => Pluralite(p)));
Console.WriteLine(Row("Dictatorial", p => Dictatorial(p, 0)));
Console.WriteLine(new string('-', 62));
Console.WriteLine();
Console.WriteLine(">>> VERDICT : aucune regle ne satisfait Pareto + IIA + Non-dictature.");
Console.WriteLine(">>> C'est exactement le theoreme d'impossibilite d'Arrow (1951).");


Refutation par force brute (216 profils, 3 alternatives, 3 votants)


--------------------------------------------------------------


Regle         Pareto     IIA (violations)       Non-dictature  


--------------------------------------------------------------


Borda         OK         ECHEC x492             OK             


Pluralite     OK         ECHEC x2568            OK             


Dictatorial   OK         OK                     Dict(V0)       


--------------------------------------------------------------


>>> VERDICT : aucune regle ne satisfait Pareto + IIA + Non-dictature.


>>> C'est exactement le theoreme d'impossibilite d'Arrow (1951).


## Synthèse

**Le théorème d'Arrow est confirmé empiriquement** par énumération exhaustive (méthode déterministe, 0 RNG) :
- **Borda** et **Pluralité** satisfont Pareto et Non-dictature mais **violent IIA** ;
- **Dictatoriale** satisfait Pareto et IIA mais **viole Non-dictature** ;
- **Aucune** des trois règles ne passe les trois axiomes à la fois — et le théorème garantit qu'**aucune règle imaginable** ne le peut (avec $\geq 3$ alternatives).

### Pourquoi le twin C# apporte-t-il quelque chose ?

1. **Déterministe** : pas de `random.seed(42)` — l'énumération est **exhaustive et reproductible**, elle ne rate aucune violation (le test stochastique Python pouvait tomber sur 0 violation par malchance) ;
2. **Pédagogique** : on **affiche le profil coupable** (contre-exemple construit section 1.2, premier exemple section 1.2-suite) plutôt qu'un simple comptage ;
3. **Parité exacte** sur le **résultat** (tableau synthétique), indépendante du RNG cross-lang.

### Lexique bilingue

| Français | English |
|----------|---------|
| Règle d'agrégation / règle de vote | Social welfare function / voting rule |
| Pareto faible (unanimité) | Weak Pareto (unanimity) |
| Indépendance aux alternatives non pertinentes | Independence of Irrelevant Alternatives (IIA) |
| Non-dictature | Non-dictatorship |
| Lemme extremal / électeur pivot | Extremal lemma / pivotal voter |

Voir le twin compagnon **`03-Voting-Methods-Csharp.ipynb`** (méthodes de vote : Pluralité, Borda, Copeland, Condorcet, IRV) pour le calcul opérationnel des gagnants.


## Exercices

> Convention (règle C.1) : les stubs s'exécutent sans erreur (`pass`/`return`/`print`). Le notebook doit tourner de bout en bout même exercices non complétés.


### Exercice 1 — Lemme extremal avec 4 alternatives

In [11]:
// Exercice 1 : Lemme extremal avec 4 alternatives
// ================================================
// Adaptez la demo de la section 2.1 avec alternatives_4 = {A, B, C, D} et la cible 'B'.
// Indice : avec 4 alternatives, B peut etre en 1re ou 4e position ; les 3 autres
// sont melangees. Generez plusieurs profils et verifiez si Borda place B en position
// extreme (1re ou 4e) dans le classement social.
var alternatives_4 = new List<string> { "A", "B", "C", "D" };
// TODO etudiant : construisez un profil ou B est toujours extreme puis appelez Borda().
object result_ex1 = null;  // TODO etudiant : votre classement social Borda
Console.WriteLine("Exercice 1 : Lemme extremal avec 4 alternatives (a completer)");


Exercice 1 : Lemme extremal avec 4 alternatives (a completer)


### Exercice 2 — Contre-exemple IIA explicite pour Borda

In [12]:
// Exercice 2 : Contre-exemple IIA explicite pour Borda
// =====================================================
// Construisez deux profils (3 votants, 3 alternatives) ou les preferences
// relatives entre A et C sont identiques pour chaque votant, mais le classement
// social Borda de A vs C change.
// Indice : changez uniquement la position de B (l'alternative "non pertinente")
// dans les preferences d'un votant pour faire basculer le verdict Borda.
var profil_1_ex2 = new List<List<string>> {
    // TODO etudiant : 3 classements
};
var profil_2_ex2 = new List<List<string>> {
    // TODO etudiant : memes contraintes A-vs-C, position de B differente
};
// TODO etudiant : verifiez (a) preferences A-vs-C identiques, (b) Borda bascule.
Console.WriteLine("Exercice 2 : Contre-exemple IIA pour Borda (a completer)");


Exercice 2 : Contre-exemple IIA pour Borda (a completer)


### Exercice 3 — Complexité de la force brute

In [13]:
// Exercice 3 : Complexite de la force brute
// ==========================================
// Implementez le nombre de profils possibles pour n votants et k alternatives.
// Indice : chaque votant a (k!) classements possibles ; pour n votants independants,
// on multiplie : (k!)^n.
public static long? CountProfiles(int nVoters, int nAlternatives)
{
    // TODO etudiant : retournez la factorielle(nAlternatives) elevee a nVoters.
    // Indice : long fact = 1; for (int i=2; i<=nAlternatives; i++) fact *= i; puis (long)Math.Pow(fact, nVoters).
    return null;  // remplacez par le calcul correct
}

Console.WriteLine("Exercice 3 : Complexite de la force brute");
var configs = new[] { (3, 3), (5, 3), (10, 4) };
foreach (var (nV, nA) in configs)
{
    var np = CountProfiles(nV, nA);
    Console.WriteLine("  (" + nV + " votants, " + nA + " alts) -> " + (np?.ToString() ?? "(a completer)") + " profils");
}


Exercice 3 : Complexite de la force brute


  (3 votants, 3 alts) -> (a completer) profils


  (5 votants, 3 alts) -> (a completer) profils


  (10 votants, 4 alts) -> (a completer) profils


### Exercice 4 — Profil de préférences paradoxaux (Condorcet vs Borda)

In [14]:
// Exercice 4 : Profil de preferences paradoxaux
// TODO etudiant : construisez un profil (3 votants, 3 alternatives) qui viole le
// critere de Condorcet avec la regle de Borda : le gagnant de Condorcet (celui qui
// bat tout autre en duel pairwise) n'est PAS le gagnant Borda.
// Indice : enumeratez les duels pairwise avec AllPrefer / IndexOf pour identifier
// le gagnant de Condorcet, puis comparez au premier de Borda(profil).
object result_ex4 = null;  // TODO etudiant : votre profil paradoxal
Console.WriteLine("Exercice a completer : Profil de preferences paradoxaux (Condorcet vs Borda)");


Exercice a completer : Profil de preferences paradoxaux (Condorcet vs Borda)


---

## Conclusion

Ce twin C# démontre le **théorème d'impossibilité d'Arrow** par voie **computo-déterministe** : énumération exhaustive des 216 profils + contre-exemples construits. La leçon clé (cf. jumeau `03-Voting-Methods-Csharp`) : **pour la parité pédagogique cross-lang d'un résultat algorithmique, le déterministe bat le stochastique** — reproductible, exhaustif, et il montre le *pourquoi* (le profil coupable) plutôt que le *combien* (un comptage opaque).

**Lien** : Marathon #4956 (jumeaux .NET ⇄ Python), EPIC #3801 axe-2 (Prong B : from-scratch). Compagnon : `03-Voting-Methods-Csharp.ipynb`.
